# Thêm thuộc tính mới vào bảng Drug

In [1]:
import sqlite3
import csv
import pandas as pd

In [4]:
conn = sqlite3.connect("../data/database/drug-food-interaction.db")
cur = conn.cursor()


In [5]:
df = pd.read_sql_query("""
    SELECT drug_name,drug_class,standardized_ingredient
    FROM Drugs
""",conn)
print("Số mẫu:", len(df))
print("Drug name unique:", df["drug_name"].nunique())
print("Drug class unique:", df["drug_class"].nunique())
print("Ingredient unique:", df["standardized_ingredient"].nunique())

Số mẫu: 2519
Drug name unique: 2519
Drug class unique: 836
Ingredient unique: 2512


In [ ]:


# 2. Thêm 2 cột mới vào bảng Drugs (nếu chưa có)
try:
    cur.execute("ALTER TABLE Drugs ADD COLUMN standardized_ingredient TEXT;")
except sqlite3.OperationalError:
    print("Cột standardized_ingredient đã tồn tại")

try:
    cur.execute("ALTER TABLE Drugs ADD COLUMN drug_class TEXT;")
except sqlite3.OperationalError:
    print("Cột drug_class đã tồn tại")

# 3. Đọc dữ liệu từ file CSV (có các cột: drug_id, drug_name, standardized_ingredient, drug_class)
with open("../data/clean_data/drugs_enriched_clean_v2.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        cur.execute("""
            UPDATE Drugs
            SET standardized_ingredient = ?,
                drug_class = ?
            WHERE drug_id = ? AND drug_name = ?;
        """, (row["standardized_ingredient"], row["drug_class"], row["drug_id"], row["drug_name"]))

# 4. Commit và đóng kết nối
conn.commit()
conn.close()


In [12]:
cur.execute("PRAGMA table_info(drugs)")
print(cur.fetchall())

[(0, 'drug_id', 'INTEGER', 0, None, 1), (1, 'drug_name', 'TEXT', 0, None, 0), (2, 'standardized_ingredient', 'TEXT', 0, None, 0), (3, 'drug_class', 'TEXT', 0, None, 0)]


In [11]:
df_new = pd.read_csv('../data/clean_data/drugs_new_only.csv')

# Nếu có drug_id thì bỏ
if "drug_id" in df_new.columns:
    df_new = df_new.drop(columns=["drug_id"])

data = list(df_new.itertuples(index=False, name=None))

print(f"Số dòng trong CSV: {len(data)}")

cur.executemany("""
INSERT OR IGNORE INTO drugs(
    drug_name,
    standardized_ingredient,
    drug_class
)
VALUES (?, ?, ?)
""", data)

conn.commit()

print(f"Số dòng thực sự được thêm: {conn.total_changes}")

Số dòng trong CSV: 1120
Số dòng thực sự được thêm: 1024


In [5]:
cur.execute("PRAGMA index_list(drugs)")
print(cur.fetchall())

[(0, 'idx_drugs_unique', 1, 'c', 0), (1, 'sqlite_autoindex_Drugs_1', 1, 'u', 0)]
